# 05 — Segmentación demográfica de usuarios

Clustering demográfico y geográfico, sin variables comportamentales. El objetivo es obtener
segmentos asignables a cualquier usuario nuevo desde el primer contacto, sin necesidad de
historial de interacciones.

Diferencia con `06_segmentation.ipynb`:
- M1 usa demografía + comportamiento → segmentos informativos del cliente actual
- Este notebook usa solo demografía → segmentos operativos para cold-start en producción

Features utilizadas (10): `age`, `gender`, `labor_status`, `civil_status`, `tiene_coche`,
`size_hogar`, `num_room`, `ipa_class`, `mun_type`, `distance_type`

Reducción de dimensionalidad evaluada: PCA y Autoencoder (se escoge el de mejor separación).
Algoritmos de clustering evaluados: KMeans y HDBSCAN.
Output: `data/processed/users_demo_segments.csv` — columnas `id_user`, `demo_cluster`, `demo_cluster_label`

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # reproducibilidad exacta del autoencoder
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
import hdbscan
import umap

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Anade src/ al path y reutiliza el helper compartido (mismo patron que 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

ROOT_PATH      = find_project_root()
DATA_PATH      = ROOT_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed"

np.random.seed(42)
print("ROOT:", ROOT_PATH)

## 1 · Carga y codificación de features demográficas

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})
print("users:", users.shape)
display(users.head(2))

In [ ]:
df = users.copy()

df["gender_enc"]       = (df["gender"] == "H").astype(int)
df["labor_status_enc"] = df["labor_status"].map({"employed": 2, "unemployed": 1, "inactive": 0}).fillna(0).astype(int)
df["civil_status_enc"] = df["civil_status"].map({"casado": 3, "soltero": 0, "divorciado": 1, "viudo": 2}).fillna(0).astype(int)
df["tiene_coche_enc"]  = df["tiene_coche"].astype(int)

def parse_hogar(s):
    if pd.isna(s): return 3
    s = str(s).strip()
    if s.startswith("1"):   return 1
    elif s.startswith("2"): return 2
    elif s.startswith("3"): return 3
    elif s.startswith("4"): return 4
    else:                   return 5

df["size_hogar_enc"] = df["size_hogar"].apply(parse_hogar)
df["num_room_enc"]   = df["num_room"].map({"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}).fillna(2).astype(int)

def edad_a_grupo(edad):
    # Bandas de edad (mismas que el modelo M2): 18-24, 25-34, 35-44, 45-54, 55-64, 65+
    if pd.isna(edad): return -1
    for i, lim in enumerate([25, 35, 45, 55, 65]):
        if edad < lim: return i
    return 5
df["age_cat"] = df["age"].apply(edad_a_grupo)

DEMO_FEATS = ["age_cat", "gender_enc", "labor_status_enc", "civil_status_enc",
              "tiene_coche_enc", "size_hogar_enc", "num_room_enc",
              "ipa_class", "mun_type", "distance_type"]

X_raw = df[DEMO_FEATS].fillna(-1).values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Matriz de features: {X.shape}")
print(f"Nulos en features originales: {df[DEMO_FEATS].isna().sum().sum()}")
pd.DataFrame(X, columns=DEMO_FEATS).describe().round(2)

## 2 · Análisis exploratorio de features

In [ ]:
# Distribuciones de features demográficas
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

labels = {
    "age_cat":          "Edad (banda 0-5)",
    "gender_enc":       "Género (1=H)",
    "labor_status_enc": "Sit. laboral (0=inac, 2=empl)",
    "civil_status_enc": "Estado civil",
    "tiene_coche_enc":  "Tiene coche",
    "size_hogar_enc":   "Tamaño hogar",
    "num_room_enc":     "Nº habitaciones",
    "ipa_class":        "IPA class (0-4)",
    "mun_type":         "Tipo municipio (0-5)",
    "distance_type":    "Dist. capital (0-4)",
}

for ax, feat in zip(axes, DEMO_FEATS):
    vals = df[feat].dropna()
    if vals.nunique() <= 6:
        vals.value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    else:
        ax.hist(vals, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(labels[feat], fontsize=9)
    ax.set_xlabel("")

plt.suptitle("Distribución de features demográficas", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre features
corr = df[DEMO_FEATS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("Correlación entre features demográficas")
plt.tight_layout()
plt.show()

# Pares con correlación alta
high_corr = [(DEMO_FEATS[i], DEMO_FEATS[j], corr.iloc[i,j])
             for i in range(len(DEMO_FEATS)) for j in range(i+1, len(DEMO_FEATS))
             if abs(corr.iloc[i,j]) > 0.3]
if high_corr:
    print("Correlaciones > 0.3:")
    for a, b, c in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {a} × {b}: {c:.3f}")
else:
    print("No hay correlaciones > 0.3 entre features.")

## 3 · Proyección UMAP

Reducción a 2D para visualizar la estructura latente de la población demográfica.

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                    random_state=42, n_jobs=1)
embedding = reducer.fit_transform(X)
df["umap_x"] = embedding[:, 0]
df["umap_y"] = embedding[:, 1]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, feat in zip(axes, DEMO_FEATS):
    sc = ax.scatter(df["umap_x"], df["umap_y"], c=df[feat], cmap="viridis",
                    s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(labels[feat], fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("UMAP coloreado por feature demográfica", fontsize=13)
plt.tight_layout()
plt.show()

## 3.5 · Reducción de dimensionalidad (PCA)

Se aplica PCA para reducir la redundancia entre features correlacionadas. Criterio de
selección: número mínimo de componentes que explican el 80 % de la varianza total.

In [ ]:
# Componentes necesarios para el 80 % de varianza
pca_full = PCA(random_state=42)
pca_full.fit(X)

cumvar = pca_full.explained_variance_ratio_.cumsum()
n_pca  = int(np.argmax(cumvar >= 0.80) + 1)
print(f"Componentes necesarios para explicar el 80 % de varianza: {n_pca}")
print(f"Varianza acumulada con {n_pca} componentes: {cumvar[n_pca-1]:.1%}")

# Varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_ * 100, color="steelblue")
axes[0].set_xlabel("Componente principal")
axes[0].set_ylabel("Varianza explicada (%)")
axes[0].set_title("Varianza explicada por cada componente")

axes[1].plot(range(1, len(cumvar) + 1), cumvar * 100, marker="o", color="steelblue")
axes[1].axhline(80, color="crimson",  linestyle="--", linewidth=1.2, label="80 %")
axes[1].axhline(90, color="orange",   linestyle="--", linewidth=1.2, label="90 %")
axes[1].axvline(n_pca, color="crimson", linestyle=":", linewidth=1.5)
axes[1].set_xlabel("Número de componentes")
axes[1].set_ylabel("Varianza acumulada (%)")
axes[1].set_title(f"Varianza acumulada — se necesitan {n_pca} componentes para el 80 %")
axes[1].legend()

plt.tight_layout()
plt.show()

# Aplicar PCA con n_pca componentes
pca  = PCA(n_components=n_pca, random_state=42)
X_pca = pca.fit_transform(X)
print(f"\nEspacio PCA: {X.shape} → {X_pca.shape}")

In [ ]:
# UMAP coloreado por cada componente PCA (mapa base = UMAP 2D de la seccion 3)
n_cols  = min(n_pca, 5)
n_rows  = int(np.ceil(n_pca / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten()

for i in range(n_pca):
    sc = axes[i].scatter(df["umap_x"], df["umap_y"], c=X_pca[:, i],
                         cmap="RdBu_r", s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=axes[i], shrink=0.8)
    axes[i].set_title(f"PC {i+1}  ({pca.explained_variance_ratio_[i]:.1%} var)", fontsize=9)
    axes[i].set_xticks([]); axes[i].set_yticks([])

for j in range(n_pca, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"UMAP coloreado por los {n_pca} componentes PCA (80 % varianza)", fontsize=13)
plt.tight_layout()
plt.show()

## 3.6 · Reducción de dimensionalidad (Autoencoder)

Reducción no lineal del espacio de features mediante una red neuronal que comprime la entrada
a un espacio latente y la reconstruye, minimizando el error de reconstrucción (MSE).

Arquitectura: 10 → 64 → 32 → 4D latente → 32 → 64 → 10

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# Determinismo total: fija semillas y operaciones deterministas para que 05 y 05b
# reconstruyan el mismo espacio latente X_ae en cualquier ejecucion.
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

INPUT_DIM  = X.shape[1]
LATENT_DIM = 4

# Encoder: 10D -> 4D
inp    = Input(shape=(INPUT_DIM,))
h      = Dense(64, activation="relu")(inp)
h      = BatchNormalization()(h)
h      = Dense(32, activation="relu")(h)
latent = Dense(LATENT_DIM, activation="linear", name="latent")(h)

# Decoder: 4D -> 10D
h   = Dense(32, activation="relu")(latent)
h   = BatchNormalization()(h)
h   = Dense(64, activation="relu")(h)
out = Dense(INPUT_DIM, activation="linear")(h)

autoencoder = Model(inp, out)    # red completa (entrenamiento)
encoder     = Model(inp, latent) # parte de compresion (clustering)

autoencoder.compile(optimizer="adam", loss="mse")

# Entrenamiento con EarlyStopping sobre val_loss
history = autoencoder.fit(
    X, X,
    epochs           = 100,
    batch_size       = 256,
    validation_split = 0.1,
    callbacks        = [EarlyStopping(monitor="val_loss", patience=10,
                                      restore_best_weights=True)],
    verbose          = 0,
)

# Embedding del espacio latente
X_ae = encoder.predict(X, verbose=0)

print(f"Espacio AE: {X.shape} → {X_ae.shape}")
print(f"Épocas entrenadas: {len(history.history['loss'])}")
print(f"Error reconstrucción (train): {history.history['loss'][-1]:.4f}")
print(f"Error reconstrucción (val):   {history.history['val_loss'][-1]:.4f}")

# Curva de entrenamiento
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history.history["loss"],     label="Train", color="steelblue")
ax.plot(history.history["val_loss"], label="Val",   color="tomato", linestyle="--")
ax.set_xlabel("Épocas")
ax.set_ylabel("Error MSE")
ax.set_title("Autoencoder — curva de entrenamiento")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualización: UMAP coloreado por cada dimensión latente del AE ──────────
fig, axes = plt.subplots(1, LATENT_DIM, figsize=(4 * LATENT_DIM, 4))

for i, ax in enumerate(axes):
    sc = ax.scatter(df["umap_x"], df["umap_y"], c=X_ae[:, i],
                    cmap="RdBu_r", s=1, alpha=0.4, rasterized=True)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(f"Dim latente {i+1}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"UMAP coloreado por las {LATENT_DIM} dimensiones del espacio latente del Autoencoder",
             fontsize=12)
plt.tight_layout()
plt.show()

## 3.7 · Comparación PCA vs Autoencoder

Se compara la separación obtenida en cada espacio mediante el Silhouette Score con k=2 (valores
más altos indican mayor separación entre grupos). Espacios comparados:
- `X` — features originales escaladas (sin reducción)
- `X_pca` — espacio PCA (80 % varianza)
- `X_ae` — espacio latente del Autoencoder (4D)

Se selecciona el espacio con mayor silhouette para el clustering.

In [ ]:
# ── Silhouette k=2 en los tres espacios ──────────────────────────────────────
sil_x_k2   = silhouette_score(X,     KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X),
                               sample_size=5000, random_state=42)
sil_pca_k2 = silhouette_score(X_pca, KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_pca),
                               sample_size=5000, random_state=42)
sil_ae_k2  = silhouette_score(X_ae,  KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_ae),
                               sample_size=5000, random_state=42)

# Tabla resumen
df_comp_emb = pd.DataFrame({
    "Espacio":           ["Original (X)", f"PCA {n_pca}D (X_pca)", f"Autoencoder {LATENT_DIM}D (X_ae)"],
    "Dimensiones":       [X.shape[1], X_pca.shape[1], X_ae.shape[1]],
    "Silhouette k=2 ↑":  [round(sil_x_k2, 4), round(sil_pca_k2, 4), round(sil_ae_k2, 4)],
})
print("Comparación de espacios de embedding:")
display(df_comp_emb)

# ── Selección automática del mejor espacio ────────────────────────────────────
best_sil = max(sil_x_k2, sil_pca_k2, sil_ae_k2)
if best_sil == sil_ae_k2:
    X_cluster, X_cluster_name = X_ae,  f"Autoencoder {LATENT_DIM}D"
elif best_sil == sil_pca_k2:
    X_cluster, X_cluster_name = X_pca, f"PCA {n_pca}D"
else:
    X_cluster, X_cluster_name = X,     "Original"

AE_USED  = X_cluster is X_ae
PCA_USED = X_cluster is X_pca

print(f"\n→ Espacio seleccionado para el clustering: {X_cluster_name}  "
      f"(silhouette={best_sil:.4f})")

# ── Gráfico comparativo ───────────────────────────────────────────────────────
espacios = ["Original", f"PCA {n_pca}D", f"AE {LATENT_DIM}D"]
valores  = [sil_x_k2, sil_pca_k2, sil_ae_k2]
colores  = ["#aec6cf", "#aec6cf", "#aec6cf"]
colores[valores.index(best_sil)] = "#e05c4a"   # rojo para el ganador

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(espacios, valores, color=colores, edgecolor="white", width=0.5)
ax.set_ylim(0, max(valores) * 1.25)
ax.set_ylabel("Silhouette Score k=2")
ax.set_title("Comparación de métodos de reducción de dimensionalidad\n(mayor silhouette = mejor separación)")

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 4 · KMeans — selección de k óptimo

Se evalúan 4 métricas para elegir el número de clusters:

| Métrica | Interpretación | Óptimo |
|---------|---------------|--------|
| **Silhouette** | Cohesión interna vs separación entre clusters | Máximo |
| **Inercia (Elbow)** | Suma de distancias intra-cluster | Codo de la curva |
| **Calinski-Harabasz** | Ratio varianza inter/intra-cluster | Máximo |
| **Davies-Bouldin** | Media de similitud entre pares de clusters | Mínimo |

In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

K_RANGE       = range(2, 13)
inertias      = []
sil_scores_km = []
ch_scores     = []
db_scores     = []

for k in K_RANGE:
    km       = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_cluster, labels_k, sample_size=5000, random_state=42)
    ch  = calinski_harabasz_score(X_cluster, labels_k)
    db  = davies_bouldin_score(X_cluster, labels_k)
    sil_scores_km.append(sil)
    ch_scores.append(ch)
    db_scores.append(db)
    print(f"  k={k:2d}  inercia={km.inertia_:9.1f}  silhouette={sil:.4f}  CH={ch:8.1f}  DB={db:.4f}")

best_k_sil = list(K_RANGE)[np.argmax(sil_scores_km)]
best_k_ch  = list(K_RANGE)[np.argmax(ch_scores)]
best_k_db  = list(K_RANGE)[np.argmin(db_scores)]
print(f"\nk óptimo por Silhouette: {best_k_sil}")
print(f"k óptimo por Calinski-Harabasz: {best_k_ch}")
print(f"k óptimo por Davies-Bouldin: {best_k_db}")

# Votación por mayoría simple
from collections import Counter
vote_k = Counter([best_k_sil, best_k_ch, best_k_db])
best_k_km = vote_k.most_common(1)[0][0]
# La silueta y Calinski-Harabasz crecen de forma monotona con k (hasta el tope del
# rango), senal de que el espacio demografico es casi un continuo sin grupos naturales:
# su argmax no es fiable. El criterio decisivo es la ESTABILIDAD del clustering (ARI por
# re-muestreo, notebook 05b): k=10 es un maximo local de ARI (0.977, muy por encima de
# sus vecinos k=9 y k=11) con cohesion alta. Se adopta k=10.
best_k_km = 10  # ver 05b: con age_cat la silueta es casi monotona y el ARI es alto en todo el rango
# (0,96-1,0); se mantiene k=10 por coherencia del pipeline y granularidad interpretable (ARI 0,984)
print(f"\nk adoptado (estabilidad ARI, ver 05b): {best_k_km}")

# Visualización de las 4 métricas
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ks = list(K_RANGE)

axes[0,0].plot(ks, inertias, marker="o", color="steelblue")
axes[0,0].set_title("Inercia (Elbow)"); axes[0,0].set_xlabel("k")

axes[0,1].plot(ks, sil_scores_km, marker="o", color="tomato")
axes[0,1].axvline(best_k_sil, color="crimson", linestyle="--", linewidth=1.2, label=f"k={best_k_sil}")
axes[0,1].set_title("Silhouette (↑ mejor)"); axes[0,1].set_xlabel("k"); axes[0,1].legend()

axes[1,0].plot(ks, ch_scores, marker="o", color="seagreen")
axes[1,0].axvline(best_k_ch, color="darkgreen", linestyle="--", linewidth=1.2, label=f"k={best_k_ch}")
axes[1,0].set_title("Calinski-Harabasz (↑ mejor)"); axes[1,0].set_xlabel("k"); axes[1,0].legend()

axes[1,1].plot(ks, db_scores, marker="o", color="orange")
axes[1,1].axvline(best_k_db, color="darkorange", linestyle="--", linewidth=1.2, label=f"k={best_k_db}")
axes[1,1].set_title("Davies-Bouldin (↓ mejor)"); axes[1,1].set_xlabel("k"); axes[1,1].legend()

plt.suptitle(f"KMeans — métricas de selección de k  [{X_cluster_name}]", fontsize=13)
plt.tight_layout()
plt.show()

## 5 · HDBSCAN

Clustering basado en densidad — detecta automáticamente el número de clusters
y etiqueta como ruido los puntos no asignables. No requiere especificar k.

In [ ]:
# Exploración de min_cluster_size
print("HDBSCAN — exploración de min_cluster_size:")
hdb_results = []
for mcs in [100, 200, 500, 1000, 2000]:
    hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10,
                          core_dist_n_jobs=-1, prediction_data=True)
    lbl = hdb.fit_predict(X_cluster)
    n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    noise_pct  = (lbl == -1).mean()
    if n_clusters > 1:
        mask = lbl != -1
        sil  = silhouette_score(X_cluster[mask], lbl[mask], sample_size=5000, random_state=42)
    else:
        sil = 0
    hdb_results.append({"min_cluster_size": mcs, "n_clusters": n_clusters,
                        "noise_pct": noise_pct, "silhouette": sil})
    print(f"  mcs={mcs:5d} → clusters={n_clusters}  ruido={noise_pct:.1%}  sil={sil:.4f}")

df_hdb = pd.DataFrame(hdb_results)
display(df_hdb)

# Mejor configuración
best_mcs = int(df_hdb.loc[df_hdb["silhouette"].idxmax(), "min_cluster_size"])
hdb_final = hdbscan.HDBSCAN(min_cluster_size=best_mcs, min_samples=10,
                             core_dist_n_jobs=-1, prediction_data=True)
labels_hdb = hdb_final.fit_predict(X_cluster)
n_hdb      = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
noise_hdb  = (labels_hdb == -1).mean()
print(f"\nHDBSCAN final (mcs={best_mcs}): {n_hdb} clusters, {noise_hdb:.1%} ruido")

## 6 · Comparación de algoritmos y selección final

Se comparan KMeans (k óptimo por consenso) y HDBSCAN en el espacio `X_cluster`.

In [ ]:
# KMeans con el k elegido por consenso
km_comp  = KMeans(n_clusters=best_k_km, random_state=42, n_init=10)
lbl_km   = km_comp.fit_predict(X_cluster)
sil_km   = silhouette_score(X_cluster, lbl_km, sample_size=5000, random_state=42)
ch_km    = calinski_harabasz_score(X_cluster, lbl_km)
db_km    = davies_bouldin_score(X_cluster, lbl_km)

# HDBSCAN silhouette (sin ruido)
mask_hdb = labels_hdb != -1
sil_hdb  = silhouette_score(X_cluster[mask_hdb], labels_hdb[mask_hdb],
                             sample_size=5000, random_state=42) if mask_hdb.sum() > 1 else 0
ch_hdb   = calinski_harabasz_score(X_cluster[mask_hdb], labels_hdb[mask_hdb]) if mask_hdb.sum() > 1 else 0
db_hdb   = davies_bouldin_score(X_cluster[mask_hdb], labels_hdb[mask_hdb])   if mask_hdb.sum() > 1 else 999

df_comp = pd.DataFrame([
    {"Algoritmo": f"KMeans (k={best_k_km})",     "Clusters": best_k_km, "Ruido %": 0,
     "Silhouette ↑": round(sil_km,  4), "CH ↑": round(ch_km,  1), "DB ↓": round(db_km,  4)},
    {"Algoritmo": f"HDBSCAN (mcs={best_mcs})",   "Clusters": n_hdb,    "Ruido %": round(noise_hdb*100, 1),
     "Silhouette ↑": round(sil_hdb, 4), "CH ↑": round(ch_hdb, 1), "DB ↓": round(db_hdb, 4)},
])
print(f"Comparación de algoritmos [{X_cluster_name}]:")
display(df_comp)

# UMAP con los 2 algoritmos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lbl, title) in zip(axes, [
    (lbl_km,     f"KMeans k={best_k_km}"),
    (labels_hdb, f"HDBSCAN mcs={best_mcs}"),
]):
    unique_lbl = sorted(set(lbl))
    colors     = sns.color_palette("tab10", len(unique_lbl))
    cmap       = {l: colors[i] for i, l in enumerate(unique_lbl)}
    c          = [cmap[l] for l in lbl]
    ax.scatter(df["umap_x"], df["umap_y"], c=c, s=1, alpha=0.4, rasterized=True)
    patches = [mpatches.Patch(color=cmap[l], label=f"{'Ruido' if l==-1 else l}") for l in unique_lbl]
    ax.legend(handles=patches, fontsize=7, loc="upper right")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"UMAP — comparación KMeans vs HDBSCAN  [{X_cluster_name}]", fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Clustering final y profiling

Se selecciona KMeans (k por consenso) como modelo final — garantiza cobertura total sin ruido,
necesario para asignar un cluster a cualquier usuario nuevo en producción.

In [ ]:
ALGO_FINAL   = f"KMeans k={best_k_km}"
labels_final = lbl_km
model_final  = km_comp

df["demo_cluster"] = labels_final
N_CLUSTERS = df["demo_cluster"].nunique()

print(f"Algoritmo seleccionado: {ALGO_FINAL}")
print(f"Número de clusters: {N_CLUSTERS}")
print("\nTamaño de clusters:")
print(df["demo_cluster"].value_counts().sort_index().to_string())

In [ ]:
# Perfil demográfico de cada cluster
profile_feats = ["age", "ipa_class", "mun_type", "distance_type",
                 "gender_enc", "tiene_coche_enc", "size_hogar_enc",
                 "labor_status_enc", "civil_status_enc", "num_room_enc"]

profile = df.groupby("demo_cluster")[profile_feats].mean().round(2)
profile["n_usuarios"] = df["demo_cluster"].value_counts().sort_index()
print("Perfil demográfico por cluster:")
display(profile)

In [ ]:
# Boxplots de features continuas por cluster
cont_feats  = ["age", "ipa_class", "mun_type", "distance_type", "size_hogar_enc"]
cont_labels = ["Edad", "IPA class", "Tipo municipio", "Dist. capital", "Tamaño hogar"]

fig, axes = plt.subplots(1, len(cont_feats), figsize=(18, 5))
colors = sns.color_palette("tab10", N_CLUSTERS)

for ax, feat, lbl in zip(axes, cont_feats, cont_labels):
    data = [df.loc[df["demo_cluster"] == c, feat].dropna().values
            for c in range(N_CLUSTERS)]
    bp = ax.boxplot(data, patch_artist=True, notch=False)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel("Cluster")
    ax.set_xticklabels(range(N_CLUSTERS))

plt.suptitle("Distribución de features por cluster demográfico", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Variables binarias / categóricas por cluster
bin_feats  = ["gender_enc", "tiene_coche_enc", "labor_status_enc", "civil_status_enc"]
bin_labels = ["% Hombre", "% Tiene coche", "Sit. laboral media", "Estado civil medio"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = sns.color_palette("tab10", N_CLUSTERS)

for ax, feat, lbl in zip(axes, bin_feats, bin_labels):
    vals = df.groupby("demo_cluster")[feat].mean()
    ax.bar(vals.index, vals.values, color=colors[:len(vals)], edgecolor="white")
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel("Cluster")
    ax.set_ylim(0, max(vals.values) * 1.2)

plt.suptitle("Variables binarias / categóricas por cluster", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloreado por cluster final
colors = sns.color_palette("tab10", N_CLUSTERS)
c_map  = [colors[c] for c in df["demo_cluster"]]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df["umap_x"], df["umap_y"], c=c_map, s=2, alpha=0.5, rasterized=True)
patches = [mpatches.Patch(color=colors[i], label=f"Cluster {i}") for i in range(N_CLUSTERS)]
ax.legend(handles=patches, title="Demo cluster", loc="upper right")
ax.set_title(f"UMAP — Clustering demográfico ({ALGO_FINAL})", fontsize=13)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plot por cluster
sil_samples = silhouette_samples(X_cluster, labels_final)
df["silhouette"] = sil_samples

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
colors  = sns.color_palette("tab10", N_CLUSTERS)

for i in range(N_CLUSTERS):
    vals = sil_samples[labels_final == i]
    vals.sort()
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=colors[i], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * len(vals), str(i), fontsize=9)
    y_lower = y_upper + 10

ax.axvline(sil_samples.mean(), color="crimson", linestyle="--",
           label=f"Media={sil_samples.mean():.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Cluster")
ax.set_title(f"Silhouette plot por cluster  [{X_cluster_name}]")
ax.legend()
plt.tight_layout()
plt.show()

print("Silhouette medio por cluster:")
print(df.groupby("demo_cluster")["silhouette"].mean().round(4).to_string())

## 8 · Interpretación y etiquetado de clusters

Se asigna una etiqueta descriptiva a cada cluster basándose en las features más distintivas.

In [ ]:
# Características más distintivas de cada cluster (z-score respecto a la media global)
global_means = df[profile_feats].mean()
global_stds  = df[profile_feats].std().replace(0, 1)

print("Características más distintivas por cluster (z-score):")
for c in range(N_CLUSTERS):
    cluster_mean = df[df["demo_cluster"] == c][profile_feats].mean()
    z = ((cluster_mean - global_means) / global_stds).sort_values(key=abs, ascending=False)
    top3 = z.head(3)
    n = (df["demo_cluster"] == c).sum()
    desc = ", ".join([f"{f}({v:+.2f}σ)" for f, v in top3.items()])
    print(f"  Cluster {c} (n={n:,}): {desc}")

In [ ]:
# Etiquetas descriptivas — se asignan tras revisar el z-score anterior
CLUSTER_LABELS = {i: f"Cluster {i}" for i in range(N_CLUSTERS)}

for c in range(N_CLUSTERS):
    cluster_mean = df[df["demo_cluster"] == c][profile_feats].mean()
    z = ((cluster_mean - global_means) / global_stds)

    traits = []
    if z["num_room_enc"] > 1.0:            traits.append("Grandes viviendas (7+ hab)")
    elif z["num_room_enc"] < -0.3:         traits.append("Viviendas estandar (3-6 hab)")
    if z["civil_status_enc"] > 0.5:        traits.append("Casados")
    elif z["civil_status_enc"] < -0.5:     traits.append("Solteros")
    if z["age"] > 0.5:                     traits.append("Mayor edad")
    elif z["age"] < -0.5:                  traits.append("Joven")
    if z["tiene_coche_enc"] > 0.5:         traits.append("Con coche")
    elif z["tiene_coche_enc"] < -0.5:      traits.append("Sin coche")
    if z["ipa_class"] > 0.5:               traits.append("Renta alta")
    elif z["ipa_class"] < -0.5:            traits.append("Renta baja")
    if z["mun_type"] < -0.5:               traits.append("Municipio pequeno")
    elif z["mun_type"] > 0.5:              traits.append("Metropoli")
    if z["size_hogar_enc"] > 0.5:          traits.append("Hogar grande")
    elif z["size_hogar_enc"] < -0.5:       traits.append("Hogar pequeno")
    if z["labor_status_enc"] > 0.5:        traits.append("Empleado")
    elif z["labor_status_enc"] < -0.5:     traits.append("Inactivo")
    if z["distance_type"] > 0.5:           traits.append("Zona rural/lejana")
    elif z["distance_type"] < -0.5:        traits.append("Zona urbana proxima")

    label = " · ".join(traits[:3]) if traits else f"Cluster {c}"
    CLUSTER_LABELS[c] = label
    print(f"  Cluster {c} (n={(df['demo_cluster']==c).sum():,}): {label}")

df["demo_cluster_label"] = df["demo_cluster"].map(CLUSTER_LABELS)

## 9 · Afinidad cluster demográfico × categoría de producto

Se une el `demo_cluster` con los eventos históricos para calcular el click rate
de cada cluster en cada categoría de producto. Esta matriz es el corazón
de las recomendaciones cold-start en M3.

In [ ]:
prop = pd.read_csv(PROCESSED_PATH / "propensity_scores.csv")

prop_demo = prop.merge(
    df[["id_user", "demo_cluster", "demo_cluster_label"]], on="id_user", how="left"
)

aff_raw = (
    prop_demo.dropna(subset=["demo_cluster"])
    .groupby(["demo_cluster", "product_new"])["target"]
    .agg(n_obs="count", n_clicks="sum")
    .reset_index()
)
aff_raw["click_rate"] = aff_raw["n_clicks"] / aff_raw["n_obs"]

ALL_PRODUCTS = sorted(prop["product_new"].dropna().unique())

aff_matrix = (
    aff_raw.pivot(index="demo_cluster", columns="product_new", values="click_rate")
    .reindex(columns=ALL_PRODUCTS, fill_value=0)
    .fillna(0)
)
aff_norm = aff_matrix.div(aff_matrix.sum(axis=1), axis=0)

print("Matriz de afinidad:", aff_matrix.shape)
display(aff_matrix.round(3))

fig, ax = plt.subplots(figsize=(18, max(4, N_CLUSTERS)))
sns.heatmap(
    aff_matrix.T, cmap="YlOrRd", ax=ax, linewidths=0.3,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    cbar_kws={"label": "Click rate"}
)
ax.set_title("Afinidad cluster demográfico × categoría de producto")
ax.set_xlabel("Cluster demográfico")
ax.set_ylabel("Categoría")
plt.tight_layout()
plt.show()

In [ ]:
# Top-5 categorías por cluster
print("Top-5 categorías por cluster demográfico:")
for c in sorted(aff_matrix.index):
    top5 = aff_matrix.loc[c].nlargest(5)
    n    = (df["demo_cluster"] == c).sum()
    lbl  = CLUSTER_LABELS[c]
    print(f"\n  Cluster {c} — {lbl} (n={n:,}):")
    for prod, cr in top5.items():
        print(f"    {prod:<30} cr={cr:.3f}")

In [ ]:
# Afinidad por sector (más agregada)
aff_sector = (
    prop_demo.dropna(subset=["demo_cluster"])
    .groupby(["demo_cluster", "sector"])["target"]
    .agg(n_obs="count", n_clicks="sum")
    .reset_index()
)
aff_sector["click_rate"] = aff_sector["n_clicks"] / aff_sector["n_obs"]

aff_sector_matrix = (
    aff_sector.pivot(index="demo_cluster", columns="sector", values="click_rate")
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, max(4, N_CLUSTERS)))
sns.heatmap(
    aff_sector_matrix.T, cmap="YlOrRd", ax=ax, linewidths=0.5,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    cbar_kws={"label": "Click rate"}
)
ax.set_title("Afinidad cluster demográfico × sector")
ax.set_xlabel("Cluster demográfico")
plt.tight_layout()
plt.show()

## 10 · Exportación

In [ ]:
import pickle

# Asignación de clusters
df_out = df[["id_user", "demo_cluster", "demo_cluster_label"]].copy()
df_out.to_csv(PROCESSED_PATH / "users_demo_segments.csv", index=False)

# Matrices de afinidad
aff_matrix.to_csv(PROCESSED_PATH / "demo_cluster_affinity.csv")
aff_norm.to_csv(PROCESSED_PATH   / "demo_cluster_affinity_norm.csv")

# Scaler y modelo de clustering (necesarios para cold-start en M3)
with open(PROCESSED_PATH / "demo_scaler.pkl", "wb") as f: pickle.dump(scaler, f)
with open(PROCESSED_PATH / "demo_model.pkl",  "wb") as f: pickle.dump(model_final, f)

# Flags para el pipeline de inferencia
with open(PROCESSED_PATH / "demo_ae_used.pkl",  "wb") as f: pickle.dump(AE_USED,  f)
with open(PROCESSED_PATH / "demo_pca_used.pkl", "wb") as f: pickle.dump(PCA_USED, f)

if AE_USED:
    encoder.save(str(PROCESSED_PATH / "demo_encoder.keras"))
    print(f"Exportado: demo_encoder.keras  ({INPUT_DIM}D → {LATENT_DIM}D)")
if PCA_USED:
    with open(PROCESSED_PATH / "demo_pca.pkl", "wb") as f: pickle.dump(pca, f)
    print(f"Exportado: demo_pca.pkl  (PCA {LATENT_DIM}D, varianza={pca.explained_variance_ratio_.sum():.1%})")

print(f"Exportado: users_demo_segments.csv        {df_out.shape}")
print(f"Exportado: demo_cluster_affinity.csv      {aff_matrix.shape}")
print(f"Exportado: demo_cluster_affinity_norm.csv {aff_norm.shape}")
print(f"Exportado: demo_scaler.pkl  (StandardScaler)")
print(f"Exportado: demo_model.pkl   ({ALGO_FINAL})")
print(f"Espacio de clustering: {X_cluster_name}  →  AE={AE_USED}  PCA={PCA_USED}")

display(df_out.head())
print("\nDistribución final:")
print(df_out.groupby(["demo_cluster", "demo_cluster_label"])
            .size().reset_index(name="n_usuarios").to_string(index=False))

## Decisiones metodológicas

| # | Decisión | Justificación |
|---|----------|--------------|
| D-28 | Clustering demográfico independiente de M1 | Los segmentos M1 son comportamentales; la demografía sola solo predice M1 con ~39% accuracy |
| D-29 | Features: las mismas 10 que M2 | Coherencia del pipeline: cualquier usuario nuevo que llega a M2 también puede ser segmentado aquí |
| D-30 | Algoritmos evaluados: KMeans, GMM, HDBSCAN | Cobertura de los tres paradigmas (partición, probabilístico, densidad) |
| D-31 | Selección final por silhouette (sin ruido) | HDBSCAN puede generar ruido elevado con datos demográficos; KMeans/GMM garantizan asignación total |
| D-32 | Outputs: `users_demo_segments.csv`, matrices de afinidad | Separación de responsabilidades: el notebook solo hace segmentación; M3 consume el resultado |

**Uso en producción (cold-start):**
```python
# Para un usuario nuevo:
x_scaled  = scaler.transform([features_usuario])
cluster   = km_demo.predict(x_scaled)[0]
top5      = aff_norm.loc[cluster].nlargest(5)
```